<a href="https://colab.research.google.com/github/columbia-data-club/meetings/blob/main/2025/march_5_data_engineering_with_polars_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![A blue background with the pandas logo and the words Columbia Data Club on it](https://raw.githubusercontent.com/columbia-data-club/meetings/main/assets/images/2025/polars.png)

# Python Data Engineering with Polars I

March 5, 2025

by [Moacir P. de Sá Pereira](https://moacir.com) for the [Columbia Data Club](https://github.com/columbia-data-club/)


This notebook provides an introduction to data engineering with [Polars](https://pola.rs). A basic understanding of Python syntax (such as the one covered in the Data Club’s [Intro to Python video](https://youtu.be/l45rzo4MUHs)) should suffice.

We will be looking to our perennial favorite today, [NYC Yellow Cab trip data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

## Pandas, OK, but Polars?

[Polars](https://pola.rs) is a library that has a lot of similar use cases to pandas. The developers present the library as “DataFrames for a new era,” and they mention a few specific reasons why it would make sense to switch to Polars, especially for intensive data engineering (with large datasets).

* Written from the ground up in Rust, so the code is very close to the machine
* This means blazing fast performance, even in comparison to PySpark and Dask, leaving pandas in its dust.
* Query optimization by using specific Polars expressions (similar to Spark)
* Lazy loading

Pandas still has a lot of use cases, of course, and probably is the better choice for row-wise operations that use methods like `.itterrows()`, but within the context of [ETL](https://en.wikipedia.org/wiki/Extract,_transform,_load), where we are transforming large datasets, we want to leverage certain optimizations that are built into Polars’s memory model.

In [18]:
import polars as pl
from datetime import datetime as dt

## Polars Syntax

Polars is built on two fundamental concepts in terms of its data wrangling engine, **contexts** and **expressions**. The latter are lazy descriptions of data transformations that make use of methods and functions built into Polars.

To me, an expression is a bit like a lambda function, down to how you can even predefine it as a transformation before you have even executed it. Here we calculate a what percentage of a taxi fare was paid as the tip:

In [4]:
tip_pct = pl.col("tip_amount") / pl.col("fare_amount")
tip_pct

<Expr ['[(col("tip_amount")) / (col("f…'] at 0x7F02FFD2EB90>

The `pl.col()` method is referencing a specific column in a dataset, but no calculations have been done yet. In fact, we have not even loaded data.

In [5]:
df = pl.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-12.parquet")

In [4]:
df.shape

(3668371, 19)

In [5]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [8]:
df.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-12-01 00:12:27,2024-12-01 00:31:12,1,9.76,1,"""N""",138,33,1,38.0,6.0,0.5,4.72,0.0,1.0,51.97,0.0,1.75
2,2024-11-30 23:56:04,2024-12-01 00:28:15,1,7.62,1,"""N""",158,42,1,37.3,1.0,0.5,8.46,0.0,1.0,50.76,2.5,0.0
2,2024-12-01 00:50:35,2024-12-01 01:24:46,4,20.07,2,"""N""",132,236,2,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75
2,2024-12-01 00:18:16,2024-12-01 00:33:16,3,2.34,1,"""N""",142,186,1,15.6,1.0,0.5,4.12,0.0,1.0,24.72,2.5,0.0
2,2024-12-01 00:56:13,2024-12-01 01:18:25,1,5.05,1,"""N""",107,80,1,26.8,1.0,0.5,5.0,0.0,1.0,36.8,2.5,0.0


In [15]:
df.select(passengers=pl.col("passenger_count").value_counts(sort=True))#.unnest("passengers")

passengers
struct[2]
"{1,2531706}"
"{2,507628}"
"{null,326291}"
"{3,133727}"
"{4,102116}"
…
"{5,22283}"
"{6,14205}"
"{8,9}"


What do these commands demonstrate to you?

## Polars Context

We have our one example of a Polars expression, and we’ll look at others in a bit, but recall that the expression needs a context. The four main contexts are:

* `select`
* `with_columns`
* `filter`
* `group_by`

Their behaviour should be somewhat familiar.

### `select`

`select` lets you choose but also aggregate certain columns.

In [26]:
df.select(
    distance = pl.col("trip_distance"),
    tip_pct = tip_pct,
    mean_tip_pct = tip_pct.mean()
).head()


distance,tip_pct,mean_tip_pct
f64,f64,f64
9.76,0.124211,NaN
7.62,0.22681,NaN
20.07,0.0,NaN
2.34,0.264103,NaN
5.05,0.186567,NaN


In [32]:
df.select(
    tip_pct = tip_pct,
    distance = pl.col("trip_distance"),
    mean_tip_pct = tip_pct.mean()
).head()

tip_pct,distance,mean_tip_pct
f64,f64,f64
0.124211,9.76,NaN
0.22681,7.62,NaN
0.0,20.07,NaN
0.264103,2.34,NaN
0.186567,5.05,NaN


In [33]:
df.select(tip_pct = tip_pct).describe()

statistic,tip_pct
str,f64
"""count""",3.668371e6
"""null_count""",0.0
"""mean""",NaN
"""std""",NaN
"""min""",-22.222222
"""25%""",-0.0
"""50%""",0.231496
"""75%""",0.286022
"""max""",inf


In [22]:
df.select(pl.col("fare_amount")).describe()

statistic,fare_amount
str,f64
"""count""",3.668371e6
"""null_count""",0.0
"""mean""",19.661293
"""std""",19.81889
"""min""",-975.0
"""25%""",9.3
"""50%""",14.2
"""75%""",23.92
"""max""",3033.1


In [23]:
df.filter(pl.col("fare_amount")== 0).select(pl.col("fare_amount"))

fare_amount
f64
0.0
0.0
0.0
0.0
0.0
…
0.0
0.0
0.0


In [24]:
df.select(fare_amount=pl.col("fare_amount")+0.001).describe()

statistic,fare_amount
str,f64
"""count""",3.668371e6
"""null_count""",0.0
"""mean""",19.662293
"""std""",19.81889
"""min""",-974.999
"""25%""",9.301
"""50%""",14.201
"""75%""",23.921
"""max""",3033.101


### `with_columns`

We can see that the `mean_tip_pct` does not get calculated correctly because the maximum tip percentage is infinity, because some people had \$0 fares. We can use `with_columns` to fix this by temporarily adding a tenth of a cent to each fare, so no fares are \$0.

`with_columns` works like `select` but returns a new data frame with all the columns as well as the newly added ones.

In [6]:
df.with_columns(
    fare_amount=pl.col("fare_amount")+0.001
).select(
    "fare_amount",
    tip_pct = tip_pct,
    distance = pl.col("trip_distance"),
    mean_tip_pct = tip_pct.mean()
).head()

fare_amount,tip_pct,distance,mean_tip_pct
f64,f64,f64,f64
38.001,0.124207,9.76,0.23526
37.301,0.226804,7.62,0.23526
70.001,0.0,20.07,0.23526
15.601,0.264086,2.34,0.23526
26.801,0.18656,5.05,0.23526


In [35]:
df_with_tip_pct = df.with_columns(
    fare_amount=pl.col("fare_amount")+0.001,
    tip_pct = tip_pct
)

In [37]:
df_with_tip_pct.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,tip_pct
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-12-01 00:12:27,2024-12-01 00:31:12,1,9.76,1,"""N""",138,33,1,38.001,6.0,0.5,4.72,0.0,1.0,51.97,0.0,1.75,0.124211
2,2024-11-30 23:56:04,2024-12-01 00:28:15,1,7.62,1,"""N""",158,42,1,37.301,1.0,0.5,8.46,0.0,1.0,50.76,2.5,0.0,0.22681
2,2024-12-01 00:50:35,2024-12-01 01:24:46,4,20.07,2,"""N""",132,236,2,70.001,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75,0.0
2,2024-12-01 00:18:16,2024-12-01 00:33:16,3,2.34,1,"""N""",142,186,1,15.601,1.0,0.5,4.12,0.0,1.0,24.72,2.5,0.0,0.264103
2,2024-12-01 00:56:13,2024-12-01 01:18:25,1,5.05,1,"""N""",107,80,1,26.801,1.0,0.5,5.0,0.0,1.0,36.8,2.5,0.0,0.186567


### `filter`

Alternatively, we could filter out the $0 fares or only adjust them using the `filter`:

In [39]:
df.filter(pl.col("fare_amount") != 0).select(
    tip_pct = tip_pct,
    distance = pl.col("trip_distance"),
    mean_tip_pct = tip_pct.mean()
).head()

tip_pct,distance,mean_tip_pct
f64,f64,f64
0.124211,9.76,0.193925
0.22681,7.62,0.193925
0.0,20.07,0.193925
0.264103,2.34,0.193925
0.186567,5.05,0.193925


Or use a new expression, `when`:

In [7]:
df.with_columns(
    fare_amount=pl.when(
        pl.col("fare_amount") == 0
    ).then(
        pl.col("fare_amount")+0.001
    ).otherwise(pl.col("fare_amount")
    )
).select(
    "fare_amount",
    tip_pct = tip_pct,
    distance = pl.col("trip_distance"),
    mean_tip_pct = tip_pct.mean()
).head()

fare_amount,tip_pct,distance,mean_tip_pct
f64,f64,f64,f64
38.0,0.124211,9.76,0.235511
37.3,0.22681,7.62,0.235511
70.0,0.0,20.07,0.235511
15.6,0.264103,2.34,0.235511
26.8,0.186567,5.05,0.235511


We can filter by date.

In [23]:
df.filter(
    pl.col("tpep_pickup_datetime").cast(pl.Date) == dt(2024, 12, 25)
).select(
    "tpep_pickup_datetime",
    pickup_time = pl.col("tpep_pickup_datetime").cast(pl.Time),
    distance = pl.col("trip_distance"),
).head()



tpep_pickup_datetime,pickup_time,distance
datetime[μs],time,f64
2024-12-25 00:03:27,00:03:27,5.94
2024-12-25 00:20:47,00:20:47,1.7
2024-12-25 00:00:48,00:00:48,1.92
2024-12-25 00:01:27,00:01:27,4.96
2024-12-25 00:27:06,00:27:06,4.3


### `group_by`